In [ ]:
import torch
# Installing my core toolkit: transformers for LLMs/Vision and diffusers for my image generation experiments.
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

In [ ]:
# Checking my environment's GPU. Generative models are heavy, so I need to verify if I have a Tesla T4 assigned for decent performance.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No GPU detected. My model runs will be significantly slower.')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("T4 GPU detected. This is a solid starting point for my experiments.")
  else:
    print("Not using a T4. I need to keep an eye on how this affects my generation speed.")

In [ ]:
# Logging into Hugging Face to access gated models like SDXL.
# I've stored my HF_TOKEN in Colab secrets to keep my workflow automated and secure.
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# My first generative output using SDXL Turbo.
# I'm using half-precision (float16) to save VRAM and using only 4 steps since 'Turbo' is optimized for speed.
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Restarting the kernel to flush system RAM. I want to make sure I have a clean environment before moving to the next model.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Now testing the full SDXL Base 1.0.
# I expect higher quality here compared to Turbo, even though it requires more steps (30) and more time.
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)

In [ ]:
# Freeing up VRAM again. This is a habit I'm building to prevent 'Out of Memory' errors during my learning sessions.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Trying the Base + Refiner ensemble.
# I'm using the Base model for the first 80% of denoising, then handing off the latents to the Refiner for the final 20% polish.
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

In [ ]:
# Cleaning up memory after the dual-model setup to keep my session healthy.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Getting the 'datasets' library so I can explore audio and text data sources for future projects.
!pip install --upgrade datasets==3.6.0

In [ ]:
# Experimenting with Text-to-Speech (TTS) using SpeechT5.
# I'm using a specific speaker embedding from the CMU Arctic dataset to control the voice characteristics.
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
# Making sure VRAM is totally clear before I attempt to load Flux, which I know is resource-intensive.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Checking for high-end hardware. For models like Flux, I really want an A100 to get the best performance.
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No GPU. Flux will likely fail or be extremely slow.')
else:
  print(gpu_info)
  if gpu_info.find('A100') >= 0:
    print("Confirmed A100. Ready to run Flux efficiently.")
  else:
    print("No A100 found. Performance might be limited.")

In [ ]:
# Running FLUX.1 [schnell]. Using bfloat16 for high precision and timing the run to track performance.
import torch
from diffusers import FluxPipeline
from IPython.display import display
from datetime import datetime

start = datetime.now()

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

display(image)

stop = datetime.now()

In [ ]:
# Calculating the cost of this run. This is a vital skill for managing compute budgets in real-world AI engineering.
seconds = (stop-start).total_seconds()
units_per_hour = 5.37
estimated_units = (units_per_hour / 3600) * seconds
estimated_cost = estimated_units * (9.99/100)
print(f"My Flux run took {seconds:.1f} seconds. Estimated compute cost: ${estimated_cost:.3f}")